# ADP 22회 - 추가 복원 통계 문제

출처: 기출문제 모음 PDF p.3: 생산 조합 확률·이항 패턴 연관성

**복원 연습 자료·예시 풀이이며 공식 정답이 아닙니다.** 원 데이터가 첨부되지 않아 동일 컬럼·분석 주제의 합성 연습 자료를 제공합니다. 수치 정답은 원 시험과 다릅니다.

동봉 data를 사용하며 원 질문·보완 조건·풀이를 분리합니다. 입력 파일은 UTF-8. 저장소 루트 또는 현재 회차 폴더에서 실행할 수 있습니다.

In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.svm import SVC
from IPython.display import display
# repository root 또는 회차 폴더에서 탐색; 경로를 하드코딩하지 않음.
ROOT=next(p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'기출문제/공통/reconstruction.py').exists())
sys.path.insert(0,str(ROOT));sys.path.insert(0,str(ROOT/'기출문제/공통'))
from reconstruction import *
from matplotlib import font_manager
font_path=ROOT/'기출문제/공통/NanumGothic-Regular.ttf'
font_manager.fontManager.addfont(str(font_path))
plt.rcParams['font.family']='NanumGothic';plt.rcParams['axes.unicode_minus']=False
HERE=ROOT/'기출문제'/'22회/보충';DATA=HERE/'data';OUT=HERE/'실행결과';OUT.mkdir(exist_ok=True)
def load(name):return pd.read_csv(DATA/name)
print('데이터 종류: 합성 복원 연습 자료, 원 시험 데이터가 아님')


데이터 종류: 합성 복원 연습 자료, 원 시험 데이터가 아님


## 1. 제품 생산 확률

자료: 데이터 없음

제품1: a,b,c 각 1개, 제품2: a,b,c 각 2개. 재료량 a=12,b=12,c=18. 생산순서 추출 확률은 원문 미정이라 각 단계 가능한 제품 중 균등 선택으로 보완.

### 3-1

재료 제약 하 제품1·제품2의 가능한 수량 조합과 최대 생산량을 구하시오. 제시한 확률 모형에서 제품1의 기대 개수·최대 가능 개수를 계산하시오.

**출처/보완:** 주제는 첨부 자료, 세부 조건·수치는 복원 연습용 보완

**풀이 근거:** 가능한 수량 범위와 확률을 혼동하지 않습니다. 원문 확률 생성 규칙이 없어 단일 정답은 정의되지 않으며 조건부 예시 기대값만 계산합니다.

In [2]:
from functools import lru_cache
feasible=[(x,y) for x in range(13) for y in range(7) if x+2*y<=12]
print('가능 조합:',feasible,'최대 총 제품:',max(x+y for x,y in feasible))
@lru_cache(None)
def expected(a,b,c):
    options=[]
    if min(a,b,c)>=1:options.append(1+expected(a-1,b-1,c-1))
    if min(a,b,c)>=2:options.append(expected(a-2,b-2,c-2))
    return float(np.mean(options)) if options else 0.
print('보완 확률하 제품1 기대개수:',expected(12,12,18),'최대 제품1:',12)

가능 조합: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4), (5, 0), (5, 1), (5, 2), (5, 3), (6, 0), (6, 1), (6, 2), (6, 3), (7, 0), (7, 1), (7, 2), (8, 0), (8, 1), (8, 2), (9, 0), (9, 1), (10, 0), (10, 1), (11, 0), (12, 0)] 최대 총 제품: 12
보완 확률하 제품1 기대개수: 4.4443359375 최대 제품1: 12


## 2. 이항 상태 순서 연관성

자료: 데이터 없음

명시 보완 순서: aabbbaaabbbabaaabbab. runs 검정으로 연속 상태의 무작위성을 검정.

### 4-1

a/b 패턴의 연속 발생이 무작위인지 검정하고, 일반적인 a/b 비율 비교와 어떻게 다른지 설명하시오.

**출처/보완:** 주제는 첨부 자료, 세부 조건·수치는 복원 연습용 보완

**풀이 근거:** 자료의 시간 순서를 검사합니다. a/b 발생 개수만 가진 chi-square는 이 연속성 귀무가설을 검사하지 않습니다.

In [3]:
from statsmodels.sandbox.stats.runs import runstest_1samp
pattern='aabbbaaabbbabaaabbab';values=np.array([x=='a' for x in pattern],dtype=int)
z,p=runstest_1samp(values,cutoff=.5,correction=True)
print({'runs':1+np.sum(values[1:]!=values[:-1]),'z':z,'p_value':p})

{'runs': np.int64(10), 'z': np.float64(-0.22973414586817037), 'p_value': np.float64(0.8182983598890985)}
